In [15]:
!pip install faiss-cpu openai


Defaulting to user installation because normal site-packages is not writeable
  Using cached anyio-4.8.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jiter-0.9.0-cp312-cp312-win_amd64.whl.metadata (5.3 kB)
  Using cached pydantic-2.10.6-py3-none-any.whl.metadata (30 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached httpcore-1.0.7-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.14.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.27.2-cp312-cp312-win_amd64.whl.metadata (6.7 kB)
   ---------------------------------------- 0.0/13.7 MB ? eta -:--:--
   ----------- ---------------------------- 3.9/13.7 MB 19.5 MB/s eta 0:00:01
   ----------------------------- ---------- 10.0/


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: C:\Users\r2com\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import json
import numpy as np
import faiss
import os
from openai import OpenAI

# 모든 JSON 데이터 로드
json_files = [f for f in os.listdir("C:/Users/r2com/Documents/GitHub/BeBaek/ChefBot/data/recipes") if f.endswith(".json")]

all_recipes = []
for file in json_files:
    with open(f"C:/Users/r2com/Documents/GitHub/BeBaek/ChefBot/data/recipes/{file}", "r", encoding="utf-8") as f:
        data = json.load(f)
        all_recipes.extend(data)

print(f"총 {len(all_recipes)} 개의 레시피 데이터 로드 완료")

# OpenAI API 키 설정
API_KEY = '' # 깃헙 반영을 위해 공백 처리  
client = OpenAI(api_key=API_KEY)

# 벡터화할 텍스트 데이터 준비
texts = [f"{recipe['name']} {recipe['ingredients']} {' '.join(recipe['recipe'])}" for recipe in all_recipes]

# OpenAI Embedding 생성 (Batch 처리)
def get_embeddings(texts, batch_size=100):
    embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        try:
            response = client.embeddings.create(
                input=batch, 
                model="text-embedding-ada-002"
            )
            batch_embeddings = [item.embedding for item in response.data] 
            embeddings.extend(batch_embeddings)
        except Exception as e:
            print(f"OpenAI API 호출 오류 발생: {e}")

    return np.array(embeddings)

# 모든 레시피 데이터 임베딩 생성
embeddings = get_embeddings(texts)

# FAISS 벡터 DB 구축
d = embeddings.shape[1]  # 벡터 차원 수
index = faiss.IndexFlatL2(d)  # L2 거리 기반 벡터 검색 인덱스
index.add(embeddings)  # 벡터 추가

# FAISS 인덱스 저장
faiss.write_index(index, "./faiss_index/recipes_faiss.index")

print("벡터화 완료 & FAISS 저장 완료")

총 511 개의 레시피 데이터 로드 완료
벡터화 완료 & FAISS 저장 완료


In [6]:
import faiss

# 저장된 FAISS 인덱스 로드
index_path = "./faiss_index/recipes_faiss.index"
index = faiss.read_index(index_path)

# 벡터 개수 확인
print(f"FAISS 인덱스 로드 완료, 저장된 벡터 개수: {index.ntotal}")

FAISS 인덱스 로드 완료, 저장된 벡터 개수: 511


In [7]:
import numpy as np
import faiss

# 저장된 FAISS 인덱스 로드
index = faiss.read_index(index_path)

# 테스트용 랜덤 벡터 생성 (벡터 차원 수 맞추기)
d = index.d  # 벡터 차원 수
test_vector = np.random.random((1, d)).astype('float32')

# 가장 가까운 3개 벡터 찾기
D, I = index.search(test_vector, 3)

print(f"검색된 벡터의 인덱스: {I[0]}")
print(f"검색된 벡터의 거리: {D[0]}")


검색된 벡터의 인덱스: [158  99  83]
검색된 벡터의 거리: [497.26575 497.32227 497.32477]


In [8]:
# 검색된 레시피 확인
search_results = [all_recipes[i] for i in I[0]]

print("검색된 레시피:")
for i, recipe in enumerate(search_results):
    print(f"{i+1}. {recipe['name']} - 재료: {recipe['ingredients']} - 레시피: {recipe['recipe']}")

검색된 레시피:
1. 막걸리 - 재료: 쌀누룩 500g,미지근한 물 500g,이스트 1작은스푼,맵쌀 500g,찹쌀 500g - 레시피: ['정보 없음']
2. 도넛·꽈배기 - 재료: 강력분 280g,박력분 70g,설탕 35g,버터 42g,소금 5g,분유 10g,인스턴트드라이이스트 8g,바닐라향 0.7g,계란 52g,물 161g,설탕 96g,계피가루 4g - 레시피: ['1. 베이킹은 시작전 전 재료를 계량및 준비를 해놓고 시작해주세요', '2. 버터 제와 모든 재료 한 번에 투입해 주시고 저속으로 믹싱합니다 한 덩어리지면 버터를 넣고 윤기있고 매끈해질때까지 믹싱하세요  (이 때 이스트는 설탕 소금과 닿이면 안됩니다)', '3. 반죽이 완성 되면 볼에 넣고 비닐로 덮어 1차 발효 합니다 3배 가량 부풀어 올라야 합니다', '4. 1차 발효후 46g 씩 분할하여 둥글리기 후 비닐 덮고 5~10분 가량 중간휴지 해주세요', '5. 반죽을 길게(35~40센치가량) 밀어펴서 양쪽끝을 잡고 꽈배기 모양으로 돌돌 말아주세요 팬닝 후 2차 발효 합니다 반죽이 살짝 부풀어 올랐다 느낌날때까지만 해주세요(대략 20분 가량) 반죽 상태를 봐가며 체크해주세요', '6. 그 사이 웍에 기름을 붓고 180~195도 사이로 온도를 맞춰 주세요', '7. 온도를 맞춘 후 기름에 넣고 1분 전후로 뒤집어 두세요 역시 뒤집고 1분 전후로 건져주세요 앞 뒤 총 2분내로 튀겨주세요', '8. 도넛을 건져낸 후 기름을 잠시 제거하며 살짝 식힌 후 따뜻할때 설탕+계피가루 섞은것을 묻혀 주세요 완성']
3. 닭갈비 - 재료: 닭가슴살 500g,양배추 1/2개,양파 1/2개,고구마 2개,깻잎 5장,떡볶이 떡 1인분,대파 조금,고추장 3T,고춧가루 3T,진간장 3T,설탕 3T,참기름 1T,다진마늘 2.5T,맛술 3T (또는 소주),후춧가루 약간,소금 조금,물 1컵 - 레시피: ['1. 1. 떡볶이 떡을 미리 불려놔요.  물에 미리 불려놔야 나중에 떡을 넣고 쪼릴때 빨리 익어요 !', '2. 2. 백종원표